---

<div style="background:rgb(1, 8, 15); color:rgb(156, 188, 239); padding:12px 14px; border-radius:6px; width:80%;">

## __To Do__:

### Adding Existing combo view to new combo. *<div style="color:green">(This is in progress, mostly done)</div>*

#### __Solution__


1. Iterate through `food_names` and `food_servings` together.
2. Split them into two groups:
   - **Regular foods** (no leading `-`)
   - **Existing combos** (leading `-`)
3. Build the regular food `UNION ALL` query using only the regular foods.
4. Build a second `UNION ALL` query that selects from the existing combo views.
5. If both queries exist, concatenate them with `UNION ALL`; otherwise, use whichever query is present.

This keeps the logic cleaner and correctly supports combos that contain other combos.

</div>

In [2]:
from wellness_tracker import run_select

def get_view_combo_names():
    """
    This function retrieves the combo names of the views in order to add to the food dropdown widget. It queries the pg_views table to get 
    the names of all views in the public schema, then constructs a SQL query that unions the distinct combo_names from each view, and 
    finally returns a list of all combo names.
    """
    query = """
        SELECT viewname
        FROM pg_views
        WHERE schemaname = 'public'
        ORDER BY schemaname, viewname;
    """
    
    try:
        views = run_select(query, return_df=True,show_errors=True)['viewname'].tolist()
        # create one SQL query that unions all the distinct combo_names from each view
        union_sql = " UNION ".join(
        f"SELECT DISTINCT combo_name AS name FROM {view}"
        for view in views
        )
        
        df = run_select(union_sql, return_df=True,show_errors=False)
        all_combo_names = df['name'].tolist()
        return all_combo_names
    except Exception as e:
        return []

    
get_view_combo_names()    

['-CC', '-FAIL', '-FFF', '-VVV', '-XXX']

In [ ]:
# ...existing code...
import ipywidgets as widgets
from IPython.display import display, HTML, Markdown
from AI import OpenAIClient
from postgres import run_select
from wellness_tracker import WellnessTracker

# Build widget data (reuse your get_view_combo_names + DB list)
all_foods = []
all_foods.extend(get_view_combo_names()) 
all_foods.extend(run_select("select name from food order by name;", return_df=True)['name'].tolist())
foods = all_foods

combo_name = widgets.Text(value="", description="Combo Name:", placeholder="Enter name")
food_search = widgets.Text(description="Search:", placeholder="Type to filter foods")
food = widgets.Dropdown(options=all_foods, description="Food:")
serving_amt = widgets.Dropdown(options=[i // 2 if i % 2 == 0 else i / 2 for i in range(1, 21)], description="Serving Size:", value=1.0)

add_food = widgets.Button(description="Add Food")
remove_last_food = widgets.Button(description="Remove Last Food")
submit = widgets.Button(description="Submit View")
out = widgets.Output()

food_list = []
serving_size_list = []
wt = WellnessTracker()

def _sync_food_options(*_):
    q = (food_search.value or "").strip().lower()
    filtered = [item for item in foods if q in str(item).lower()]
    food.options = filtered
    food.value = filtered[0] if filtered else None

food_search.observe(_sync_food_options, names="value")
_sync_food_options()

def on_add_food_clicked(_):
   
    # check if added food is a duplicate
    if food_list.count(food.value) > 0:
        display(Markdown(f"Duplicate Food: {food.value} has already been added."))
    
    # for proper submissions 
    if len(food_list) < 10:
          food_list.append(food.value)
          serving_size_list.append(serving_amt.value)

          with out:
           display(Markdown(f"**Food Added:** {food.value} — {serving_amt.value} serving(s)\n<br>"))   
              
    else:
            with out:
                display(Markdown("Maximum of 10 foods allowed. Remove a food or submit the Combo as is."))
            
     
    

def on_remove_last_food_clicked(_):
    if not food_list:
        with out:
            display(Markdown("No foods to remove."))
        return
    removed_food = food_list.pop()
    if serving_size_list:
        removed_amt = serving_size_list.pop()
    with out:
        display(Markdown(f"**Removed:** {removed_food} — {removed_amt if 'removed_amt' in locals() else '?'}"))

def on_submit(_):
    
     # check if combo has at least two foods
    if len(food_list) < 2:
        display(Markdown("__Food Combos need at least two foods.__"))
        return
    submit.disabled = True
    try:
        wt.create_combo_view(combo_name.value, food_list, serving_size_list)
        out.clear_output(wait=True)
        items_html = "".join(f"<li>{f} — {q}</li>" for f, q in zip(food_list, serving_size_list))
        with out:
            display(HTML(f"""
                <div style="background:rgb(1,8,15);color:teal;padding:10px;border-radius:9px;width:80%;">
                  <h2 style="background-color:rgb(210,220,230);color:rgb(20,19,25);margin-top:0;padding:5px;border-radius:5px;width:fit-content;">
                    Combo Created
                  </h2>
                  <p><b>Name:</b> {combo_name.value}</p>
                  <ul style="margin:0;padding-left:1.2em;">{items_html}</ul>
                </div>
            """))
            # clear the lists to allow for submitting another view
            food_list.clear()
            serving_size_list.clear()
            # clear widget values
            combo_name.value = ""
            serving_amt.value = 1
    except Exception as e:
        out.clear_output(wait=True)
        with out:
            display(HTML("<br><b><div style='color: red;'>DATABASE ERROR:</b></div>"))
            display(HTML(f"<div style='background: rgb(230, 230, 230); padding:3px;border-radius:5px;font-family: Courier;'>{str(e)}</div><br>"))
        ai = OpenAIClient()
        sf = combo_name.value
        sf_items = ", ".join(f"{f} ({q})" for f, q in zip(food_list, serving_size_list)) or "<None>"
        prompt = (f"Explain concisely why creating this combo failed.\nCombo name: {sf}\nItems: {sf_items}\n\nException:\n{e}")
        ai_msg = ai.get_response(prompt)
        with out:
            display(HTML(f"<div style='background-color:rgb(22, 22, 22); padding:5px;border-radius:5px;width:fit-content;color:#fff;'>{ai_msg}</div>"))
    finally:
        submit.disabled = False

# Prevent duplicate handlers on re-run
add_food._click_handlers.callbacks.clear()
remove_last_food._click_handlers.callbacks.clear()
submit._click_handlers.callbacks.clear()

add_food.on_click(on_add_food_clicked)
remove_last_food.on_click(on_remove_last_food_clicked)
submit.on_click(on_submit)

display(combo_name, food_search, food, serving_amt, add_food, remove_last_food, submit, out)
# ...existing code...

Text(value='', description='Combo Name:', placeholder='Enter name')

Text(value='', description='Search:', placeholder='Type to filter foods')

Dropdown(description='Food:', options=('-AA', '-CC', '-FAIL', '-FFF', '-VVV', '-XXX', 'Almond Milk', 'Almonds'…

Dropdown(description='Serving Size:', index=1, options=(0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5, 5.5, 6, 6.5, 7…

Button(description='Add Food', style=ButtonStyle())

Button(description='Remove Last Food', style=ButtonStyle())

Button(description='Submit View', style=ButtonStyle())

Output()

Duplicate Food: Butter has already been added.